
# Implementación: Construyendo el "Cerebro" en Código (SimpleRNN)

En la sesión anterior, calculamos manualmente cómo fluye la información paso a paso. Ahora vamos a **profesionalizar** esa idea construyendo nuestra propia clase de Red Neuronal Recurrente usando **Python + NumPy**.

**Objetivo:** crear un objeto reusable (estilo PyTorch / TensorFlow), pero hecho por nosotros, para entender exactamente:

- cómo se actualiza la **memoria** (`h_t`),
- cómo se obtiene una **predicción** (`y`),
- y por qué entrenar una RNN requiere una idea especial: **Backpropagation Through Time (BPTT)**.

> Nota para BI: aquí no buscamos “la mejor RNN”, sino entender el *mecanismo* que permite modelar secuencias.



## 1. La Estructura de la Clase

Vamos a definir una clase `SimpleRNN`. Necesitaremos dos métodos principales:

- `__init__`: inicializa pesos y sesgos (bias).
- `forward`: ejecuta el *bucle temporal* (el “paso hacia adelante”).

### Convenciones de forma (shapes)

Para que no haya ambigüedad, fijamos formas explícitas:

- Entrada en cada tiempo: `x_t` tiene forma `(input_size, 1)`
- Memoria: `h_t` tiene forma `(hidden_size, 1)`
- Salida: `y` tiene forma `(output_size, 1)`


In [1]:

import numpy as np


class SimpleRNN:
    """
    A minimal Recurrent Neural Network (RNN) implemented with NumPy.

    This class focuses on the forward pass to illustrate how the hidden state
    (memory) evolves through time for sequential data.

    Notation:
        h_t = tanh(W_xh x_t + W_hh h_{t-1} + b_h)
        y   = W_hy h_T + b_y
    """

    def __init__(self, input_size: int, hidden_size: int, output_size: int, seed: int | None = 42) -> None:
        """
        Initialize the RNN parameters.

        Args:
            input_size: Dimension of the input vector at each time step.
            hidden_size: Size of the hidden state (memory).
            output_size: Dimension of the output vector.
            seed: Random seed for reproducibility (optional).
        """
        self.input_size = int(input_size)
        self.hidden_size = int(hidden_size)
        self.output_size = int(output_size)

        if seed is not None:
            rng = np.random.default_rng(seed)
            normal = rng.standard_normal
        else:
            normal = np.random.randn

        # Small random initialization (helps avoid saturation at start)
        self.W_xh = normal((self.hidden_size, self.input_size)) * 0.01  # Input -> Hidden
        self.W_hh = normal((self.hidden_size, self.hidden_size)) * 0.01  # Hidden -> Hidden (memory)
        self.W_hy = normal((self.output_size, self.hidden_size)) * 0.01  # Hidden -> Output

        self.bh = np.zeros((self.hidden_size, 1))  # Hidden bias
        self.by = np.zeros((self.output_size, 1))  # Output bias

        # Caches for inspection (filled after forward pass)
        self.last_inputs: list[np.ndarray] = []
        self.hidden_states: dict[int, np.ndarray] = {}

    @staticmethod
    def _as_column_vector(x: float | int | np.ndarray, size: int) -> np.ndarray:
        """
        Convert a scalar or 1D array to a column vector of shape (size, 1).

        Args:
            x: Scalar or array-like input.
            size: Expected length of the vector.

        Returns:
            Column vector with shape (size, 1).
        """
        arr = np.asarray(x, dtype=float).reshape(-1, 1)
        if arr.shape != (size, 1):
            raise ValueError(f"Expected shape ({size}, 1), got {arr.shape}.")
        return arr

    def forward(self, inputs: list[float] | list[np.ndarray]) -> tuple[np.ndarray, np.ndarray]:
        """
        Forward pass through time.

        Args:
            inputs: A sequence of inputs. Each element can be a scalar (if input_size=1)
                    or a vector-like object that matches (input_size, 1).

        Returns:
            y: Output vector of shape (output_size, 1).
            h: Final hidden state of shape (hidden_size, 1).
        """
        # Initialize hidden state (memory) with zeros
        h = np.zeros((self.hidden_size, 1))

        # Store for later inspection
        self.last_inputs = []
        self.hidden_states = {0: h.copy()}

        for t, x_t in enumerate(inputs, start=1):
            x_col = self._as_column_vector(x_t, self.input_size)
            self.last_inputs.append(x_col)

            # Core RNN equation: h_t = tanh(W_xh x_t + W_hh h_{t-1} + b_h)
            h = np.tanh(self.W_xh @ x_col + self.W_hh @ h + self.bh)

            self.hidden_states[t] = h.copy()

        # Final output computed from the last hidden state
        y = self.W_hy @ h + self.by
        return y, h


print("✅ SimpleRNN class defined successfully.")


✅ SimpleRNN class defined successfully.



## 2. Probando la Red con los Mismos Datos (y los mismos pesos)

Vamos a usar la secuencia `[1, 2, 3]`.

Para replicar exactamente el ejercicio manual, vamos a:

1) Crear la red con `input_size=1`, `hidden_size=1`, `output_size=1`  
2) “Forzar” los pesos para que coincidan con el ejemplo:

- `W_xh = 0.5`
- `W_hh = 0.8`
- `W_hy = 1.0`
- `b_h = 0`, `b_y = 0`

**Resultado esperado:** salida ≈ `0.975`.


In [2]:

# 1) Instantiate the RNN
rnn = SimpleRNN(input_size=1, hidden_size=1, output_size=1, seed=123)

# 2) Overwrite weights to match the manual example
rnn.W_xh = np.array([[0.5]])
rnn.W_hh = np.array([[0.8]])
rnn.W_hy = np.array([[1.0]])
rnn.bh = np.array([[0.0]])
rnn.by = np.array([[0.0]])

print("✅ Weights manually set to replicate the paper-and-pencil example.")

# 3) Input sequence
inputs = [1, 2, 3]

# 4) Forward pass
y_pred, h_final = rnn.forward(inputs)

print("-" * 40)
print(f"Input sequence: {inputs}")
print(f"Final prediction y: {float(y_pred[0, 0]):.4f}")
print(f"Final hidden state h_T: {float(h_final[0, 0]):.4f}")
print("-" * 40)


✅ Weights manually set to replicate the paper-and-pencil example.
----------------------------------------
Input sequence: [1, 2, 3]
Final prediction y: 0.9759
Final hidden state h_T: 0.9759
----------------------------------------



## 3. Visualizando la “Memoria” Paso a Paso

Una gran ventaja de programar nuestra RNN es que podemos inspeccionar qué estaba “pensando” en cada tiempo.

Después del `forward`, la red guarda los estados ocultos en `rnn.hidden_states`:

- `t=0` es la memoria inicial (en blanco)
- `t=1..T` son las memorias actualizadas luego de cada entrada

Vamos a imprimir el rastro completo.


In [3]:

print("🧠 Memory trace (hidden states):")
for t in range(len(inputs) + 1):
    h_t = float(rnn.hidden_states[t][0, 0])
    if t == 0:
        print(f"t={t:>2} (initial)              -> h_{t} = {h_t:.3f}")
    else:
        x_t = inputs[t - 1]
        print(f"t={t:>2} (after seeing x={x_t}) -> h_{t} = {h_t:.3f}")


🧠 Memory trace (hidden states):
t= 0 (initial)              -> h_0 = 0.000
t= 1 (after seeing x=1) -> h_1 = 0.462
t= 2 (after seeing x=2) -> h_2 = 0.879
t= 3 (after seeing x=3) -> h_3 = 0.976



## 4. El Concepto de "Backpropagation Through Time" (BPTT)

Ya sabemos ejecutar la red **hacia adelante** (forward). La pregunta natural es:

👉 **¿Cómo se ajustan los pesos para mejorar la predicción?**

En una red feedforward, el gradiente viaja hacia atrás una sola vez por capa.

En una RNN, el mismo peso (por ejemplo `W_hh`) se reutiliza en **cada paso temporal**:

- `W_hh` se usa en `t=1`
- se vuelve a usar en `t=2`
- y otra vez en `t=3`
- … y así sucesivamente

### Intuición del BPTT

1) Calculamos el error en la salida final (por ejemplo, en `t=T`)  
2) “Desenrollamos” la RNN como si fuera una red profunda de **T capas** (una por tiempo)  
3) El error se propaga hacia atrás: `T → T-1 → ... → 1`  
4) En cada paso se acumula “culpa” (gradiente) para los **mismos pesos compartidos**  
5) Actualizamos los pesos **una sola vez**, con la suma de contribuciones de todos los tiempos

### Analogía rápida

Es como corregir un procedimiento donde el resultado final salió mal:

- el error se ve al final,
- pero pudo originarse en un paso anterior,
- y hay que rastrear la responsabilidad hacia atrás para corregir todo el proceso.



## 5. Cierre y lo que sigue

En este notebook ya logramos:

- encapsular una RNN en una clase reusable,
- replicar el cálculo manual con pesos forzados,
- inspeccionar el rastro de memoria `h_t` paso a paso,
- entender por qué el entrenamiento requiere BPTT.

En el siguiente notebook vamos a ver el “dolor real”:

- por qué el gradiente se desvanece o explota en secuencias largas,
- y cómo **LSTM / GRU** nacen como respuesta práctica a ese problema.
